In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 105 samples and 15151 columns.
First few rows of the data:


,overall survival period from diagnosis date (year),overall survival event,overall survival period from resection date (year),overall survival event.1,Sample,gender,age at diagnosis (years),mutation category,subtype,primary tumor location,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
0,4.1863,0.0,4.0822,0.0,BAIX,M,48.0,BRAF,SunExposed,right occipital lobe (scalp),...,4.238683,5.644566,6.233697,1.819664,2.430716,5.270749,1.157515,5.129748,6.262725,6.327488
1,3.4575,0.0,2.1808,0.0,BAY,M,63.0,WT,SunExposed,right lateral arm,...,3.369043,5.044671,5.661581,2.281543,2.968554,4.835667,1.755134,4.864588,4.854396,5.872085
2,5.1616,0.0,1.3425,0.0,BILI,M,39.0,WT,SunExposed,left ear (tragus),...,3.831359,4.721497,4.685914,2.305708,2.726233,4.244033,0.718545,4.291452,4.525052,5.248131
3,9.3562,0.0,3.3534,0.0,BROT,M,71.0,WT,SunExposed,right shoulder,...,4.318865,5.368309,5.091451,1.646034,2.601106,3.944951,0.920962,4.061010,7.017117,5.386654
4,8.6767,1.0,0.2795,1.0,CHER,M,55.0,RAS,SunExposed,right arm,...,3.278025,4.431412,5.114594,2.005626,2.485849,4.602899,0.851397,4.473105,7.566240,5.258804


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for overall survival period from diagnosis date (year) or overall survival event: 4
Number of samples with complete data for both variables: 101
Number of censored samples (event_var == 0.0): 52
Number of events (event_var == 1.0): 49
Censored ratio (among complete): 52 / 101 = 0.515 (51.5%)
Number of samples with incomplete data for overall survival period from resection date (year) or overall survival event: 4
Number of samples with complete data for both variables: 101
Number of censored samples (event_var == 0.0): 52
Number of events (event_var == 1.0): 49
Censored ratio (among complete): 52 / 101 = 0.515 (51.5%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE162682/description.json
